<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg" align="left">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">Supervivencia en Cáncer de Pulmón</p>
<p style="margin: 0; text-align:right;">2026-1 · Máster universitario en Ciencia de datos (Data science)</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Notebook adaptado a NSCLC ctDx MSK 2022</p>
</div>
</div>
<div style="width:100%; height:80px; clear:both;"></div>

<div class="alert alert-block alert-info">
<strong>Nombre y apellidos: JULIO ÚBEDA QUESADA</strong>
</div>

# Análisis de Supervivencia y Predicción de Riesgo Clínico en Cáncer de Pulmón No Microcítico

**Notebook específico para el conjunto de datos NSCLC ctDx MSK 2022 (`nsclc_ctdx_msk_2022_clinical_data.tsv`).**

Este cuaderno replica la lógica utilizada en el notebook de METABRIC y la adapta a la cohorte NSCLC ctDx MSK 2022:

1. carga y auditoría del dataset clínico;
2. EDA reutilizando el módulo `EDA_functions.py`;
3. control de granularidad muestra-paciente;
4. limpieza orientada a supervivencia;
5. preprocesamiento reproducible para Kaplan-Meier, Cox, Random Survival Forest y DeepSurv.

La unidad final de modelado será el **paciente**, no la muestra, para evitar que el mismo individuo aparezca simultáneamente en train/test o tenga un peso artificialmente mayor.

Para la correcta ejecución del cuaderno necesitaremos importar los siguientes módulos. Las librerías específicas de supervivencia se importan de forma tolerante: si alguna no está instalada, el preprocesamiento puede ejecutarse igualmente y se mostrará un aviso.

In [ ]:
# Sistema
import os
import sys
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Ciencia de datos
import numpy as np
import pandas as pd

# Estadística
from scipy import stats
from scipy.stats import wilcoxon
from scipy.stats.mstats import winsorize

# Visualización
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import seaborn as sns

# Machine Learning / preprocesamiento
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Persistencia de objetos preprocesados
import joblib

# ── Imports opcionales para modelado de supervivencia ───────────────────────

try:
    from lifelines import KaplanMeierFitter, CoxPHFitter
    from lifelines.statistics import logrank_test, multivariate_logrank_test
    HAS_LIFELINES = True
except ImportError:
    HAS_LIFELINES = False
    KaplanMeierFitter = None
    CoxPHFitter = None
    logrank_test = None
    multivariate_logrank_test = None

try:
    from sksurv.util import Surv
    from sksurv.ensemble import RandomSurvivalForest
    HAS_SKSURV = True
except ImportError:
    HAS_SKSURV = False
    Surv = None
    RandomSurvivalForest = None

# DeepSurv puede implementarse con pycox/torchtuples u otras librerías.
# Aquí solo dejamos los arrays preparados.
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

print(f"lifelines disponible       : {HAS_LIFELINES}")
print(f"scikit-survival disponible : {HAS_SKSURV}")
print(f"torch disponible           : {HAS_TORCH}")

In [ ]:
# Importa y actualiza las funciones personalizadas de EDA.
# Compatible tanto con estructura de proyecto:
#   ├── notebooks/NSCLC_CTdx_MSK_2022.ipynb
#   └── utils/EDA_functions.py
# como con el notebook y EDA_functions.py en la misma carpeta.

import importlib

candidate_paths = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve().parent / "utils",
    Path.cwd().resolve() / "utils",
    Path("/mnt/data"),  # útil en este entorno de generación/validación
]

for p in candidate_paths:
    if p.exists() and str(p) not in sys.path:
        sys.path.append(str(p))

try:
    import utils.EDA_functions as eda
except ModuleNotFoundError:
    import EDA_functions as eda

eda = importlib.reload(eda)

print("✓ Módulo EDA importado correctamente.")
print("Funciones reutilizadas:", [f for f in dir(eda) if not f.startswith("_") and callable(getattr(eda, f))])

---
# 1. Estrategia de adquisición de datos

Se trabaja con el archivo clínico público de cBioPortal/MSK:

* **NSCLC ctDx MSK 2022:** `nsclc_ctdx_msk_2022_clinical_data.tsv`

El objetivo del notebook no es entrenar todavía los modelos finales, sino dejar una matriz de covariables y objetivos de supervivencia lista para:

* **Kaplan-Meier:** análisis no paramétrico por grupos clínicos.
* **Cox PH / Cox penalizado:** modelos semiparamétricos.
* **Random Survival Forest:** modelo no lineal basado en árboles.
* **DeepSurv:** red neuronal para riesgo proporcional.

La variable de tiempo principal será `Overall Survival (Months)` y el evento se extraerá de `Overall Survival Status`.

---
# 2. Cargar los datos clínicos

La celda siguiente busca el archivo en varias rutas habituales. Si se ejecuta desde el proyecto, la ruta esperada es `../data/nsclc_ctdx_msk_2022_clinical_data.tsv`. Si se ejecuta en la misma carpeta que el TSV, también funcionará sin cambios.

In [ ]:
DATASET_NAME = "NSCLC-ctDx-MSK-2022"
DATA_FILE = "nsclc_ctdx_msk_2022_clinical_data.tsv"

DATA_PATH_CANDIDATES = [
    Path("../data") / DATA_FILE,
    Path("data") / DATA_FILE,
    Path(DATA_FILE),
    Path("/mnt/data") / DATA_FILE,  # útil en este entorno de generación/validación
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"No se encontró el archivo {DATA_FILE}. "
        "Colócalo en ../data/, en ./data/ o en la misma carpeta del notebook."
    )

nsclc_raw = pd.read_csv(DATA_PATH, sep="\t")

print(f"✓ Dataset cargado desde: {DATA_PATH}")
print(f"Dimensiones: {nsclc_raw.shape[0]} filas × {nsclc_raw.shape[1]} columnas")
nsclc_raw.head()

## 2.1. Normalización de columnas numéricas mixtas

Algunas columnas procedentes de cBioPortal se leen como texto porque contienen valores especiales como `>90`, porcentajes (`20%`) o etiquetas no numéricas (`default`). Antes del EDA se crean/ajustan versiones numéricas para evitar errores durante el preprocesamiento.

In [ ]:
def parse_mixed_numeric(series):
    """Convierte series con valores como '>90', '20%' o 'default' a numérico."""
    s = series.astype("string").str.strip()
    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
        "None": np.nan,
        "default": np.nan,
        "Default": np.nan,
    })
    s = s.str.replace("%", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"^>\s*", "", regex=True)
    s = s.str.replace(r"^<\s*", "", regex=True)
    return pd.to_numeric(s, errors="coerce")

nsclc = nsclc_raw.copy()

if "Age at Which Sequencing was Reported (Years)" in nsclc.columns:
    nsclc["Age at Which Sequencing was Reported (Years)"] = parse_mixed_numeric(
        nsclc["Age at Which Sequencing was Reported (Years)"]
    )

if "Tumor Purity" in nsclc.columns:
    nsclc["Tumor Purity Numeric"] = parse_mixed_numeric(nsclc["Tumor Purity"])

print("Columnas con normalización numérica:")
display(
    nsclc[[
        c for c in [
            "Age at Which Sequencing was Reported (Years)",
            "Tumor Purity",
            "Tumor Purity Numeric"
        ] if c in nsclc.columns
    ]].head(10)
)

---
# **3. NSCLC ctDx MSK 2022**

## **3.1. Descripción del conjunto de datos**

La tabla combina información clínica, histológica, de muestra y de métricas genómicas agregadas. A diferencia de METABRIC, el dataset está a **nivel muestra**, por lo que varios pacientes pueden aparecer repetidos.

Puntos críticos para supervivencia:

* `Overall Survival (Months)` define el tiempo de seguimiento.
* `Overall Survival Status` codifica el evento (`1:DECEASED`) o censura (`0:LIVING`).
* `Patient ID` debe usarse para deduplicar antes de dividir train/test.
* Variables como `Sample ID`, `Gene Panel`, `Sample Class` o `Site` describen adquisición/procesamiento y deben tratarse con cautela porque pueden introducir efectos de lote.

In [ ]:
# Resumen general reutilizando el módulo EDA
eda.describe_df(nsclc)

In [ ]:
# Resumen de valores nulos
eda.null_summary(nsclc)

In [ ]:
# Duplicados por paciente y granularidad muestra-paciente
n_patients = nsclc["Patient ID"].nunique()
n_samples = nsclc["Sample ID"].nunique()
n_duplicated_patient_rows = nsclc["Patient ID"].duplicated().sum()

print(f"Pacientes únicos : {n_patients}")
print(f"Muestras únicas  : {n_samples}")
print(f"Filas con Patient ID duplicado: {n_duplicated_patient_rows}")

target_consistency = (
    nsclc
    .groupby("Patient ID")[["Overall Survival (Months)", "Overall Survival Status"]]
    .nunique(dropna=False)
)

n_inconsistent_target = (
    (target_consistency["Overall Survival (Months)"] > 1) |
    (target_consistency["Overall Survival Status"] > 1)
).sum()

print(f"Pacientes con objetivo de supervivencia inconsistente entre muestras: {n_inconsistent_target}")

nsclc.loc[
    nsclc["Patient ID"].duplicated(keep=False),
    [
        "Patient ID", "Sample ID", "Sample Class", "Sample Type", "Gene Panel",
        "Overall Survival (Months)", "Overall Survival Status"
    ]
].sort_values(["Patient ID", "Sample ID"]).head(30)

**Decisión de granularidad:** para los modelos de supervivencia se utilizará una única fila por paciente. En esta cohorte hay múltiples muestras por paciente, pero el endpoint de supervivencia es paciente-específico. Más adelante se deduplicará tras filtrar a NSCLC y se priorizarán muestras con mayor cobertura genómica.

## **3.2. Análisis estadístico básico**

El EDA se organiza por familias lógicas de variables. Las funciones gráficas se reutilizan desde `EDA_functions.py`:

* `plot_categorical_subplots`
* `plot_numerical_subplots`
* `plot_correlation_heatmap`

Los gráficos se guardarán en HTML/PNG para poder consultarlos fuera del notebook.

In [ ]:
# Directorios de salida para los gráficos de EDA
# Se prioriza la estructura del proyecto (../images/EDA/NSCLC_CTdx_MSK_2022) y,
# si no hay permisos, se usa una carpeta local junto al notebook.
preferred_eda_output = Path("../images/EDA/NSCLC_CTdx_MSK_2022")
fallback_eda_output = Path("images/EDA/NSCLC_CTdx_MSK_2022")

try:
    preferred_eda_output.mkdir(parents=True, exist_ok=True)
    EDA_OUTPUT_PATH = preferred_eda_output
except OSError as exc:
    print(f"⚠ No se pudo crear {preferred_eda_output} ({exc}). Se usará {fallback_eda_output}.")
    fallback_eda_output.mkdir(parents=True, exist_ok=True)
    EDA_OUTPUT_PATH = fallback_eda_output

print(f"Los gráficos HTML/PNG se guardarán en: {EDA_OUTPUT_PATH.resolve()}")

### **3.2.1. Análisis de Variables Categóricas: Distribución y Tendencias**

#### **A. Identificadores, diagnóstico y metadatos del estudio**

Estas variables ayudan a validar procedencia, subtipo tumoral y homogeneidad de la cohorte. No todas serán covariables finales.

In [ ]:
eda.plot_categorical_subplots(
    df=nsclc.loc[:, nsclc.columns.isin([
        "Study ID",
        "Cancer Type",
        "Cancer Type Detailed",
        "Oncotree Code",
        "Gene Panel",
        "Sample Class",
        "Sample Type",
        "Site",
        "Successful ctDx Lung"
    ])],
    group_name="Identificadores Diagnóstico y Metadatos del Estudio",
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    ncol=3
)

#### **B. Perfil clínico-demográfico**

Incluye edad, sexo, raza, etnicidad y hábitos tabáquicos. Estas variables son candidatas naturales para describir la cohorte y ajustar modelos multivariantes.

In [ ]:
eda.plot_categorical_subplots(
    df=nsclc.loc[:, nsclc.columns.isin([
        "Sex",
        "Race Category",
        "Ethnicity Category",
        "Smoking Status",
        "Prior Treatment",
        "Age Greater than Median"
    ])],
    group_name="Perfil Clínico-Demográfico",
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    ncol=3
)

#### **C. Caracterización histológica, anatómica y de enfermedad**

Agrupa variables de histología, localización tumoral, sitio metastásico y estadio al momento de la extracción.

In [ ]:
eda.plot_categorical_subplots(
    df=nsclc.loc[:, nsclc.columns.isin([
        "Histology",
        "Primary Tumor Site",
        "Metastatic Site",
        "Extrapulmonary",
        "MSI Type"
    ])],
    group_name="Caracterización Histológica Anatómica y Molecular",
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    ncol=3
)

#### **D. Variables de estado de supervivencia**

El estado de supervivencia se utilizará exclusivamente para construir el indicador `event`. No debe entrar como covariable.

In [ ]:
eda.plot_categorical_subplots(
    df=nsclc.loc[:, nsclc.columns.isin([
        "Overall Survival Status"
    ])],
    group_name="Variables de Estado de Supervivencia",
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    ncol=2
)

### **3.2.2. Análisis de Variables Numéricas: Distribución y Tendencias**

#### **A. Edad y estructura de seguimiento**

Se revisan edades disponibles y la variable de supervivencia global.

In [ ]:
eda.plot_numerical_subplots(
    df=nsclc.loc[:, nsclc.columns.isin([
        "Age at Which Sequencing was Reported (Years)",
        "Patient Current Age",
        "Overall Survival (Months)"
    ])],
    group_name="Edad y Supervivencia Global",
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    ncol=3
)

#### **B. Métricas genómicas y moleculares agregadas**

Estas variables resumen carga mutacional, inestabilidad y pureza tumoral. Se evaluará su completitud antes del modelado.

In [ ]:
eda.plot_numerical_subplots(
    df=nsclc.loc[:, nsclc.columns.isin([
        "Fraction Genome Altered",
        "MSI Score",
        "Mutation Count",
        "TMB (nonsynonymous)",
        "Tumor Purity Numeric"
    ])],
    group_name="Métricas Genómicas y Moleculares",
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    ncol=3
)

#### **C. Carga tumoral, estadio y estructura muestral**

`Metabolic Tumor Volume` puede ser clínicamente útil, pero suele tener muchos valores perdidos. `Number of Samples Per Patient` es una variable de auditoría y no se usará como predictor final.

In [ ]:
eda.plot_numerical_subplots(
    df=nsclc.loc[:, nsclc.columns.isin([
        "Metabolic Tumor Volume",
        "Stage at Draw",
        "Number of Samples Per Patient"
    ])],
    group_name="Carga Tumoral Estadio y Estructura Muestral",
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    ncol=3
)

### **3.2.3. Análisis de Correlaciones**

Se calcula un mapa de correlaciones sobre variables numéricas relevantes. Este paso ayuda a detectar redundancias y relaciones potencialmente problemáticas antes de Cox.

In [ ]:
NUMERIC_CORR_COLS = [
    "Age at Which Sequencing was Reported (Years)",
    "Patient Current Age",
    "Overall Survival (Months)",
    "Fraction Genome Altered",
    "MSI Score",
    "Mutation Count",
    "TMB (nonsynonymous)",
    "Tumor Purity Numeric",
    "Metabolic Tumor Volume",
    "Stage at Draw",
]

eda.plot_correlation_heatmap(
    df=nsclc.loc[:, nsclc.columns.isin(NUMERIC_CORR_COLS)],
    dataset_name=DATASET_NAME,
    output_path=str(EDA_OUTPUT_PATH),
    top_n=8
)

**Lectura esperada del heatmap antes del modelado:**

* `Mutation Count` y `TMB (nonsynonymous)` pueden estar correlacionadas porque ambas resumen carga mutacional.
* `Patient Current Age` puede no representar una covariable basal estricta, por lo que se excluirá del pipeline principal para evitar proxies de seguimiento.
* Variables con alta ausencia se revisarán con un umbral explícito antes de construir la matriz final.

---
# **3.3. Preprocesamiento orientado a modelos de supervivencia**

A partir de aquí se construyen los objetos finales para modelado:

* `km_df`: tabla compacta para Kaplan-Meier.
* `cox_train_df`, `cox_test_df`: DataFrames para Cox PH.
* `X_train`, `X_test`, `y_train`, `y_test`: entradas para Random Survival Forest.
* `X_train_deepsurv`, `X_test_deepsurv`: arrays `float32` para DeepSurv.

In [ ]:
# Trabajamos sobre una copia para no modificar el DataFrame del EDA
nsclc_prep = nsclc.copy()

print(f"Shape inicial: {nsclc_prep.shape}")

### **3.3.1. Filtrado explícito a NSCLC**

El archivo contiene algunas filas con otros diagnósticos, normalmente por muestras múltiples o anotaciones alternativas. Para este notebook se mantiene la cohorte principal de **Non-Small Cell Lung Cancer**.

In [ ]:
KEEP_NSCLC_ONLY = True

if KEEP_NSCLC_ONLY and "Cancer Type" in nsclc_prep.columns:
    n_antes = len(nsclc_prep)
    nsclc_prep = nsclc_prep[
        nsclc_prep["Cancer Type"].eq("Non-Small Cell Lung Cancer")
    ].copy()

    print(f"Registros eliminados fuera de NSCLC: {n_antes - len(nsclc_prep)}")
    print(f"Shape tras filtrar NSCLC: {nsclc_prep.shape}")
    display(nsclc_prep["Cancer Type"].value_counts(dropna=False).to_frame("n"))
else:
    print("No se aplicó filtrado por Cancer Type.")

### **3.3.2. Selección de una muestra por paciente**

El endpoint de supervivencia es paciente-específico. Para evitar duplicados, se selecciona una fila por `Patient ID`.

Criterio por defecto:

1. priorizar `Tumor` frente a `cfDNA`, porque maximiza la completitud de métricas genómicas agregadas como FGA/TMB/MSI;
2. priorizar `Primary` frente a `Metastasis`, si existe;
3. elegir de forma determinista por `Sample ID`.

Si el análisis se quisiera restringir estrictamente a biopsia líquida, basta con invertir `SAMPLE_CLASS_PRIORITY`.

In [ ]:
n_antes = len(nsclc_prep)

SAMPLE_CLASS_PRIORITY = {
    "Tumor": 0,
    "cfDNA": 1,
}

SAMPLE_TYPE_PRIORITY = {
    "Primary": 0,
    "Metastasis": 1,
    "Local Recurrence": 2,
    "Unknown": 3,
}

if "Patient ID" in nsclc_prep.columns:
    nsclc_prep["__sample_class_priority"] = (
        nsclc_prep.get("Sample Class", pd.Series(index=nsclc_prep.index, dtype=object))
        .map(SAMPLE_CLASS_PRIORITY)
        .fillna(9)
    )
    nsclc_prep["__sample_type_priority"] = (
        nsclc_prep.get("Sample Type", pd.Series(index=nsclc_prep.index, dtype=object))
        .map(SAMPLE_TYPE_PRIORITY)
        .fillna(9)
    )

    sort_cols = ["Patient ID", "__sample_class_priority", "__sample_type_priority"]
    if "Sample ID" in nsclc_prep.columns:
        sort_cols.append("Sample ID")

    nsclc_prep = (
        nsclc_prep
        .sort_values(sort_cols)
        .drop_duplicates(subset=["Patient ID"], keep="first")
        .drop(columns=["__sample_class_priority", "__sample_type_priority"])
    )

print(f"Registros eliminados por duplicidad muestra-paciente: {n_antes - len(nsclc_prep)}")
print(f"Shape resultante: {nsclc_prep.shape}")

for col in ["Sample Class", "Sample Type", "Gene Panel"]:
    if col in nsclc_prep.columns:
        print(f"\nDistribución de {col} tras deduplicar:")
        display(nsclc_prep[col].value_counts(dropna=False).to_frame("n"))

### **3.3.3. Eliminación de columnas de metadatos, identificadores y estructura muestral**

Se eliminan identificadores únicos, metadatos de estudio y variables de laboratorio/muestra que no representan una característica clínica basal del paciente.

In [ ]:
COLS_DROP_METADATA = [
    # Identificadores únicos o casi únicos
    "Patient ID",
    "Sample ID",
    "Patient Display Name",

    # Metadatos de estudio
    "Study ID",

    # Tras filtrar a NSCLC, estas columnas son constantes o redundantes con Histology
    "Cancer Type",
    "Cancer Type Detailed",
    "Oncotree Code",

    # Variables de adquisición/procesamiento de muestra
    "Gene Panel",
    "Sample Class",
    "Sample Type",
    "Number of Samples Per Patient",
    "Site",
    "Successful ctDx Lung",

    # Variables anatómicas redundantes o de baja variabilidad tras filtrar NSCLC
    "Primary Tumor Site",

    # Columna textual original; se conserva Tumor Purity Numeric
    "Tumor Purity",
]

cols_present = [c for c in COLS_DROP_METADATA if c in nsclc_prep.columns]
nsclc_prep.drop(columns=COLS_DROP_METADATA, inplace=True, errors="ignore")

print(f"Columnas eliminadas ({len(cols_present)}): {cols_present}")
print(f"Shape resultante: {nsclc_prep.shape}")

### **3.3.4. Eliminación de variables redundantes, con posible fuga o baja completitud**

Se excluyen variables que duplican información, pueden codificar seguimiento posterior o presentan demasiada ausencia para un pipeline basal robusto.

In [ ]:
COLS_REDUNDANTES = [
    # Edad actual puede ser proxy de seguimiento/estado vital; se prefiere edad en secuenciación
    "Patient Current Age",

    # Redundante con edad continua
    "Age Greater than Median",
]

COLS_LEAKAGE = [
    # No hay columnas de DFS en este dataset, pero se deja la lista por trazabilidad
]

COLS_REDUNDANTES_LEAKAGE = COLS_REDUNDANTES + COLS_LEAKAGE

cols_present = [c for c in COLS_REDUNDANTES_LEAKAGE if c in nsclc_prep.columns]
nsclc_prep.drop(columns=COLS_REDUNDANTES_LEAKAGE, inplace=True, errors="ignore")

print(f"Columnas eliminadas por redundancia/fuga ({len(cols_present)}): {cols_present}")
print(f"Shape resultante: {nsclc_prep.shape}")

# Eliminación por alto porcentaje de nulos, sin tocar columnas del objetivo bruto.
RAW_TARGET_COLS = ["Overall Survival (Months)", "Overall Survival Status"]
HIGH_MISSING_THRESHOLD = 0.85

missing_rate = nsclc_prep.drop(columns=RAW_TARGET_COLS, errors="ignore").isna().mean()
cols_high_missing = missing_rate[missing_rate > HIGH_MISSING_THRESHOLD].index.tolist()

nsclc_prep.drop(columns=cols_high_missing, inplace=True, errors="ignore")

print(f"\nUmbral de nulos aplicado: > {HIGH_MISSING_THRESHOLD:.0%}")
print(f"Columnas eliminadas por alta ausencia ({len(cols_high_missing)}): {cols_high_missing}")
print(f"Shape resultante: {nsclc_prep.shape}")

# Eliminación de columnas constantes o casi vacías tras filtrado/deduplicación.
constant_cols = [
    c for c in nsclc_prep.columns
    if c not in RAW_TARGET_COLS and nsclc_prep[c].nunique(dropna=True) <= 1
]

nsclc_prep.drop(columns=constant_cols, inplace=True, errors="ignore")

print(f"\nColumnas constantes eliminadas ({len(constant_cols)}): {constant_cols}")
print(f"Shape resultante: {nsclc_prep.shape}")

### **3.3.5. Definición y parsing de las variables objetivo (`duration` y `event`)**

Todos los modelos de supervivencia requieren:

* `duration`: tiempo de seguimiento en meses.
* `event`: indicador binario, donde `1` indica muerte observada y `0` censura.

La codificación se extrae de `Overall Survival Status`.

In [ ]:
nsclc_prep["duration"] = pd.to_numeric(
    nsclc_prep["Overall Survival (Months)"],
    errors="coerce"
)

nsclc_prep["event"] = (
    nsclc_prep["Overall Survival Status"]
    .astype(str)
    .str.extract(r"^(\d)")[0]
    .astype(float)
)

TARGET_COLS = ["duration", "event"]

# Verificación del objetivo
eda.describe_df(nsclc_prep[TARGET_COLS])

In [ ]:
print(f"Columnas objetivo creadas ({len(TARGET_COLS)}): {TARGET_COLS}")
print(f"Shape resultante: {nsclc_prep.shape}")

print("\nDistribución del evento:")
display(nsclc_prep["event"].value_counts(dropna=False).to_frame("n"))

event_rate = nsclc_prep["event"].mean(skipna=True)
print(f"Tasa de eventos observada: {event_rate:.2%}")

### **3.3.6. Eliminación de registros con valores nulos en las variables objetivo**

Imputar tiempo de supervivencia o estado vital no es metodológicamente aceptable. Los registros sin objetivo completo se eliminan.

In [ ]:
n_antes = len(nsclc_prep)

nsclc_prep.dropna(subset=["duration", "event"], inplace=True)

COLS_POST_TARGET = [
    "Overall Survival (Months)",
    "Overall Survival Status",
]

nsclc_prep.drop(columns=COLS_POST_TARGET, inplace=True, errors="ignore")

print(f"Columnas eliminadas tras crear target ({len(COLS_POST_TARGET)}): {COLS_POST_TARGET}")
print(f"Shape resultante: {nsclc_prep.shape}")
print(f"\nRegistros eliminados: {n_antes - len(nsclc_prep)} ({(n_antes - len(nsclc_prep)) / n_antes:.2%})")
print(f"Registros restantes : {len(nsclc_prep)}")

### **3.3.7. Tratamiento de tiempos de supervivencia en cero**

Los tiempos `T = 0` pueden causar problemas numéricos en modelos basados en Cox y en algunos algoritmos de supervivencia. Se corrigen a un valor pequeño positivo.

In [ ]:
EPSILON = 0.001

n_ceros = (nsclc_prep["duration"] == 0).sum()
nsclc_prep["duration"] = nsclc_prep["duration"].clip(lower=EPSILON)

print(f"Registros con T=0 corregidos : {n_ceros}")
print(f"Tiempo mínimo tras corrección: {nsclc_prep['duration'].min():.4f} meses")
print(f"Tiempo máximo                : {nsclc_prep['duration'].max():.2f} meses")

### **3.3.8. Dataset auxiliar para Kaplan-Meier**

Kaplan-Meier no requiere matriz de covariables codificada. Para los análisis estratificados se conserva una tabla compacta con el objetivo y algunas variables clínicas interpretables.

In [ ]:
KM_GROUP_COLS = [
    "Histology",
    "Sex",
    "Race Category",
    "Ethnicity Category",
    "Smoking Status",
    "Prior Treatment",
    "Extrapulmonary",
    "Metastatic Site",
    "Age at Which Sequencing was Reported (Years)",
    "TMB (nonsynonymous)",
    "Mutation Count",
]

km_cols_present = TARGET_COLS + [c for c in KM_GROUP_COLS if c in nsclc_prep.columns]
km_df = nsclc_prep[km_cols_present].copy()

if "Age at Which Sequencing was Reported (Years)" in km_df.columns:
    km_df["Age Group"] = pd.cut(
        km_df["Age at Which Sequencing was Reported (Years)"],
        bins=[0, 50, 65, np.inf],
        labels=["<50", "50-65", ">65"],
        right=False
    )

if "TMB (nonsynonymous)" in km_df.columns and km_df["TMB (nonsynonymous)"].notna().sum() > 0:
    tmb_median = km_df["TMB (nonsynonymous)"].median()
    km_df["TMB Group"] = np.where(
        km_df["TMB (nonsynonymous)"].isna(),
        "Unknown",
        np.where(km_df["TMB (nonsynonymous)"] >= tmb_median, "TMB alto", "TMB bajo")
    )

print(f"Shape km_df: {km_df.shape}")
km_df.head()

### **3.3.9. Selección del subconjunto de covariables para el modelado**

Una vez eliminado el objetivo original y las columnas con fuga de información, las covariables restantes quedan como candidatas. Las variables categóricas se codificarán mediante One-Hot Encoding y las numéricas se imputarán/escalarán.

In [ ]:
FEATURE_COLS = [c for c in nsclc_prep.columns if c not in TARGET_COLS]

print(f"Covariables candidatas ({len(FEATURE_COLS)}):")
for c in FEATURE_COLS:
    print(f"  - {c}")

print(f"\nShape resultante: {nsclc_prep.shape}")

### **3.3.10. División estratificada en conjuntos train/test**

Se usa una partición 80/20 estratificada por evento. Esto mantiene una proporción similar de fallecimientos/censuras en ambos conjuntos y evita optimismo por fuga entre muestras del mismo paciente.

In [ ]:
X = nsclc_prep[FEATURE_COLS].copy()
y_duration = nsclc_prep["duration"].astype(float).values
y_event = nsclc_prep["event"].astype(int).values

X_train, X_test, dur_train, dur_test, evt_train, evt_test = train_test_split(
    X,
    y_duration,
    y_event,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_event
)

print(f"Train: {len(X_train)} registros | Tasa de eventos: {evt_train.mean():.2%}")
print(f"Test : {len(X_test)} registros | Tasa de eventos: {evt_test.mean():.2%}")

### **3.3.11. Imputación de valores nulos en covariables**

La estrategia reproduce el enfoque usado en METABRIC:

* Variables numéricas → mediana calculada solo en train.
* Variables categóricas → categoría explícita `Unknown`.

La imputación se ajusta únicamente en el conjunto de entrenamiento para evitar fuga de información.

In [ ]:
NUM_COLS = X_train.select_dtypes(include="number").columns.tolist()
CAT_COLS = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Variables numéricas ({len(NUM_COLS)}): {NUM_COLS}")
print(f"\nVariables categóricas ({len(CAT_COLS)}): {CAT_COLS}")

In [ ]:
imputer_num = SimpleImputer(strategy="median")
imputer_cat = SimpleImputer(strategy="constant", fill_value="Unknown")

if NUM_COLS:
    X_train[NUM_COLS] = imputer_num.fit_transform(X_train[NUM_COLS])
    X_test[NUM_COLS] = imputer_num.transform(X_test[NUM_COLS])

if CAT_COLS:
    X_train[CAT_COLS] = imputer_cat.fit_transform(X_train[CAT_COLS])
    X_test[CAT_COLS] = imputer_cat.transform(X_test[CAT_COLS])

    # Evita mezcla de tipos boolean/string tras imputación
    X_train[CAT_COLS] = X_train[CAT_COLS].astype(str)
    X_test[CAT_COLS] = X_test[CAT_COLS].astype(str)

print("Nulos tras imputación:")
print(f"  X_train: {X_train.isna().sum().sum()}")
print(f"  X_test : {X_test.isna().sum().sum()}")

### **3.3.12. Agrupación de categorías raras**

Para reducir inestabilidad en Cox y evitar columnas one-hot extremadamente escasas, se agrupan como `Other` las categorías con menos de 10 observaciones en train. La regla se aprende en train y se aplica después a test.

In [ ]:
MIN_CATEGORY_COUNT = 10
rare_category_maps = {}

for col in CAT_COLS:
    counts = X_train[col].value_counts(dropna=False)
    keep_categories = counts[counts >= MIN_CATEGORY_COUNT].index.astype(str).tolist()
    rare_category_maps[col] = keep_categories

    X_train[col] = np.where(X_train[col].isin(keep_categories), X_train[col], "Other")
    X_test[col] = np.where(X_test[col].isin(keep_categories), X_test[col], "Other")

print(f"Categorías raras agrupadas con mínimo de {MIN_CATEGORY_COUNT} observaciones en train.")

for col, cats in rare_category_maps.items():
    print(f"{col}: {len(cats)} categorías conservadas")

### **3.3.13. Tratamiento de outliers en variables numéricas continuas**

Se aplica winsorización empírica al percentil 1–99, calculando los umbrales únicamente en train y aplicándolos a train/test. Esto estabiliza Cox y DeepSurv sin usar información del test.

In [ ]:
COLS_WINSORIZE = [
    "Age at Which Sequencing was Reported (Years)",
    "Fraction Genome Altered",
    "MSI Score",
    "Mutation Count",
    "TMB (nonsynonymous)",
    "Tumor Purity Numeric",
    "Metabolic Tumor Volume",
]

winsor_limits = {}

for col in COLS_WINSORIZE:
    if col in X_train.columns:
        p01 = np.percentile(X_train[col].astype(float), 1)
        p99 = np.percentile(X_train[col].astype(float), 99)
        winsor_limits[col] = (p01, p99)

        X_train[col] = X_train[col].clip(lower=p01, upper=p99)
        X_test[col] = X_test[col].clip(lower=p01, upper=p99)

        print(f"{col:<45} -> clipped a [{p01:.4f}, {p99:.4f}]")

### **3.3.14. Codificación de variables categóricas**

Cox, RSF y DeepSurv requieren entradas numéricas. Se usa One-Hot Encoding con:

* `drop='first'` para reducir multicolinealidad perfecta.
* `handle_unknown='ignore'` para categorías ausentes en train pero presentes en test.

In [ ]:
try:
    encoder = OneHotEncoder(
        drop="first",
        sparse_output=False,
        handle_unknown="ignore",
        dtype=float
    )
except TypeError:
    # Compatibilidad con versiones antiguas de scikit-learn
    encoder = OneHotEncoder(
        drop="first",
        sparse=False,
        handle_unknown="ignore",
        dtype=float
    )

if CAT_COLS:
    # Ajustar y transformar
    ohe_train = encoder.fit_transform(X_train[CAT_COLS])
    ohe_test = encoder.transform(X_test[CAT_COLS])

    # Nombres de las nuevas columnas
    ohe_feature_names = encoder.get_feature_names_out(CAT_COLS).tolist()

    # Construir DataFrames con columnas OHE
    ohe_train_df = pd.DataFrame(ohe_train, columns=ohe_feature_names, index=X_train.index)
    ohe_test_df = pd.DataFrame(ohe_test, columns=ohe_feature_names, index=X_test.index)

    # Reemplazar categóricas originales por OHE
    X_train = pd.concat([X_train.drop(columns=CAT_COLS), ohe_train_df], axis=1)
    X_test = pd.concat([X_test.drop(columns=CAT_COLS), ohe_test_df], axis=1)
else:
    ohe_feature_names = []

print(f"Shape X_train tras OHE: {X_train.shape}")
print(f"Shape X_test tras OHE : {X_test.shape}")
print(f"Total covariables finales: {X_train.shape[1]}")

### **3.3.15. Escalado de variables numéricas**

Se aplica `StandardScaler` solo sobre las variables numéricas originales. Aunque RSF no requiere escalado, mantener un pipeline homogéneo facilita Cox penalizado y DeepSurv.

In [ ]:
scaler = StandardScaler()

if NUM_COLS:
    X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
    X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

print(f"Variables escaladas ({len(NUM_COLS)}): {NUM_COLS}")

if NUM_COLS:
    display(eda.describe_df(X_train[NUM_COLS])[["Column", "mean", "std"]])

### **3.3.16. Creación del `structured array` de scikit-survival**

`scikit-survival` espera un array estructurado con dos campos:

* `event`: booleano.
* `time`: tiempo de seguimiento.

Si `scikit-survival` no está instalado, se crea manualmente una estructura equivalente.

In [ ]:
if HAS_SKSURV and Surv is not None:
    y_train = Surv.from_arrays(
        event=evt_train.astype(bool),
        time=dur_train.astype(float)
    )

    y_test = Surv.from_arrays(
        event=evt_test.astype(bool),
        time=dur_test.astype(float)
    )
else:
    y_train = np.array(
        list(zip(evt_train.astype(bool), dur_train.astype(float))),
        dtype=[("event", "?"), ("time", "<f8")]
    )

    y_test = np.array(
        list(zip(evt_test.astype(bool), dur_test.astype(float))),
        dtype=[("event", "?"), ("time", "<f8")]
    )

print(f"y_train dtype: {y_train.dtype} | shape: {y_train.shape}")
print(f"y_test dtype : {y_test.dtype} | shape: {y_test.shape}")

### **3.3.17. Objetos finales para Cox, RSF y DeepSurv**

Se crean vistas específicas para cada familia de modelos:

* `cox_train_df`, `cox_test_df`: formato cómodo para `lifelines.CoxPHFitter`.
* `X_train_np`, `X_test_np`, `y_train`, `y_test`: formato para `scikit-survival`.
* `X_train_deepsurv`, `X_test_deepsurv`: tensores/arrays de entrada para DeepSurv.

In [ ]:
# Arrays numpy generales
X_train_np = X_train.values.astype(float)
X_test_np = X_test.values.astype(float)

# DataFrames para lifelines / CoxPHFitter
cox_train_df = X_train.copy()
cox_train_df["duration"] = dur_train.astype(float)
cox_train_df["event"] = evt_train.astype(int)

cox_test_df = X_test.copy()
cox_test_df["duration"] = dur_test.astype(float)
cox_test_df["event"] = evt_test.astype(int)

# Arrays para DeepSurv / pycox
X_train_deepsurv = X_train_np.astype("float32")
X_test_deepsurv = X_test_np.astype("float32")
dur_train_deepsurv = dur_train.astype("float32")
dur_test_deepsurv = dur_test.astype("float32")
evt_train_deepsurv = evt_train.astype("float32")
evt_test_deepsurv = evt_test.astype("float32")

print(f"X_train_np        : {X_train_np.shape}")
print(f"X_test_np         : {X_test_np.shape}")
print(f"cox_train_df      : {cox_train_df.shape}")
print(f"cox_test_df       : {cox_test_df.shape}")
print(f"X_train_deepsurv  : {X_train_deepsurv.shape} | dtype: {X_train_deepsurv.dtype}")
print(f"X_test_deepsurv   : {X_test_deepsurv.shape} | dtype: {X_test_deepsurv.dtype}")

print("\nValidación de nulos:")
print(f"  X_train: {np.isnan(X_train_np).sum()}")
print(f"  X_test : {np.isnan(X_test_np).sum()}")

### **3.3.18. Comprobaciones finales de coherencia**

Antes de entrenar modelos, se comprueba:

* igualdad de columnas entre train/test;
* ausencia de valores nulos o infinitos;
* duración positiva;
* resumen de eventos y número final de covariables.

In [ ]:
assert list(X_train.columns) == list(X_test.columns), "Train/test no tienen las mismas columnas."
assert np.isfinite(X_train_np).all(), "X_train contiene valores no finitos."
assert np.isfinite(X_test_np).all(), "X_test contiene valores no finitos."
assert (dur_train > 0).all() and (dur_test > 0).all(), "Hay duraciones no positivas."

summary_preproc = pd.DataFrame({
    "split": ["train", "test"],
    "n": [len(X_train), len(X_test)],
    "events": [int(evt_train.sum()), int(evt_test.sum())],
    "event_rate": [evt_train.mean(), evt_test.mean()],
    "median_duration": [np.median(dur_train), np.median(dur_test)],
    "n_features": [X_train.shape[1], X_test.shape[1]]
})

summary_preproc

### **3.3.19. Guardado opcional de objetos preprocesados**

Esta celda guarda los objetos necesarios para reutilizarlos en notebooks de modelado. Si no quieres persistir nada, puedes omitirla.

In [ ]:
preferred_processed_output = Path("../data/processed")
fallback_processed_output = Path("data/processed")

try:
    preferred_processed_output.mkdir(parents=True, exist_ok=True)
    PROCESSED_OUTPUT_PATH = preferred_processed_output
except OSError as exc:
    print(f"⚠ No se pudo crear {preferred_processed_output} ({exc}). Se usará {fallback_processed_output}.")
    fallback_processed_output.mkdir(parents=True, exist_ok=True)
    PROCESSED_OUTPUT_PATH = fallback_processed_output

processed_bundle = {
    "dataset_name": DATASET_NAME,
    "random_state": RANDOM_STATE,
    "feature_cols_raw": FEATURE_COLS,
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
    "ohe_feature_names": ohe_feature_names,
    "final_feature_names": X_train.columns.tolist(),
    "winsor_limits": winsor_limits,
    "rare_category_maps": rare_category_maps,
    "imputer_num": imputer_num,
    "imputer_cat": imputer_cat,
    "encoder": encoder,
    "scaler": scaler,
    "km_df": km_df,
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "dur_train": dur_train,
    "dur_test": dur_test,
    "evt_train": evt_train,
    "evt_test": evt_test,
    "cox_train_df": cox_train_df,
    "cox_test_df": cox_test_df,
    "X_train_deepsurv": X_train_deepsurv,
    "X_test_deepsurv": X_test_deepsurv,
    "dur_train_deepsurv": dur_train_deepsurv,
    "dur_test_deepsurv": dur_test_deepsurv,
    "evt_train_deepsurv": evt_train_deepsurv,
    "evt_test_deepsurv": evt_test_deepsurv,
    "summary_preproc": summary_preproc,
}

bundle_path = PROCESSED_OUTPUT_PATH / "nsclc_ctdx_msk_2022_survival_preprocessed.joblib"
features_path = PROCESSED_OUTPUT_PATH / "nsclc_ctdx_msk_2022_final_features.csv"
summary_path = PROCESSED_OUTPUT_PATH / "nsclc_ctdx_msk_2022_preprocessing_summary.csv"

joblib.dump(processed_bundle, bundle_path)
pd.Series(X_train.columns, name="feature").to_csv(features_path, index=False)
summary_preproc.to_csv(summary_path, index=False)

print(f"✓ Bundle guardado en : {bundle_path.resolve()}")
print(f"✓ Features guardadas: {features_path.resolve()}")
print(f"✓ Resumen guardado   : {summary_path.resolve()}")

---
# 4. Siguientes pasos de modelado

Con este preprocesamiento ya están disponibles los objetos necesarios para entrenar y evaluar:

* **Kaplan-Meier:** usar `km_df` para curvas por `Histology`, `Smoking Status`, `Prior Treatment`, `Age Group` o `TMB Group`.
* **Cox PH:** usar `cox_train_df` y `cox_test_df`.
* **Random Survival Forest:** usar `X_train`, `X_test`, `y_train`, `y_test`.
* **DeepSurv:** usar `X_train_deepsurv`, `dur_train_deepsurv`, `evt_train_deepsurv` y sus equivalentes de test.

Antes de entrenar modelos definitivos conviene revisar si se quiere mantener el criterio de deduplicación `Tumor > cfDNA` o si, por hipótesis biológica, se prefiere restringir el análisis a `cfDNA`.